<a href="https://colab.research.google.com/github/RJRM999/EDA-DS-Projects/blob/main/LangGraph_Project_Maths_Fns.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [56]:
!pip install -q langgraph langchain langchain-groq

Gemnini

In [57]:
import os
from getpass import getpass

os.environ["GROQ_API_KEY"] = getpass("Enter your Groq API key: ")

Enter your Groq API key: ··········


In [58]:
from langchain_groq import ChatGroq
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage, SystemMessage
from langgraph.graph import StateGraph, START, MessagesState
from langgraph. prebuilt import ToolNode, tools_condition

Tools defined

In [59]:
@tool
def add(a:float, b:float) -> float:
    """Add two numbers"""
    return a+b

@tool
def subtract(a:float, b:float) -> float:
    """Subtract b from a"""
    return a-b

@tool
def multiply(a:float, b:float) -> float:
    """Multiply two numbers"""
    return a * b

@tool
def divide(a:float, b:float) -> str:
    """Divide a by b. Handles division by zero."""
    if b== 0:
        return "Error: Division by zero is not allowed."
    return str(a/b)


Gemini Model and bind tools

In [68]:
tools = [add, subtract, multiply, divide]

llm = ChatGroq(model = "llama-3.3-70b-versatile", temperature = 0)

llm_with_tools = llm.bind_tools(tools, tool_choice ="auto")

Node Definition

In [69]:
def mathbot(state: MessagesState):
    system_message = SystemMessage(

        content="""

        You are a helpful assistant.

        For mathematical operations, always use the available tools:
        - add for addition
        - subtract for subtraction
        - multiply for multiplication
        - divide for division

        For general questions, answer directly.
        """

    )

    response = llm_with_tools.invoke([system_message] + state["messages"])

    return {"messages": [response]}



Graph build

In [70]:
graph_builder = StateGraph(MessagesState)

graph_builder.add_node('mathbot', mathbot)
graph_builder. add_node("tools", ToolNode(tools))

graph_builder.add_edge(START, 'mathbot')

graph_builder.add_conditional_edges("mathbot", tools_condition)

graph_builder.add_edge("tools", "mathbot")

graph = graph_builder.compile()


Helper Function

In [71]:
def run_agent(user_query):
  result = graph.invoke({"messages":[HumanMessage(content = user_query)]})

  return result["messages"][-1].content


Testing with queries

In [72]:
print(run_agent("What is 5 plus 3?"))

The answer is 8.


In [73]:
print(run_agent("What is 10 divided by 2?"))

The result of 10 divided by 2 is 5.0.


In [74]:
print(run_agent("What is 6 multiplied by 9?"))

The result of 6 multiplied by 9 is 54.


In [75]:
print(run_agent("What is 6 multiplied by 9?"))

The result of 6 multiplied by 9 is 54.


In [76]:
print(run_agent("Explain machine learning in simple words."))

Machine learning is a way to teach computers to learn and improve on their own, without being explicitly programmed for every task. It's like teaching a child to recognize dogs and cats - you show them many pictures, and they start to understand what makes a dog a dog and a cat a cat. The computer uses data, like these pictures, to learn patterns and make decisions on its own, getting better and better over time.


In [77]:
print(run_agent("Who is Geoffrey Hinton?"))

Geoffrey Hinton is a British-Canadian cognitive psychologist and computer scientist, best known for his work on artificial neural networks and deep learning. He is often referred to as the "Godfather of Deep Learning" due to his significant contributions to the field. Hinton is a professor at the University of Toronto and a chief scientific advisor at the Vector Institute for Artificial Intelligence. He has made major contributions to the development of backpropagation, Boltzmann machines, and other neural network architectures, and has won numerous awards for his work, including the Turing Award.


In [78]:
print(run_agent("Who invented backpropagation technique?"))

The backpropagation technique was invented by David Rumelhart, Geoffrey Hinton, and Ronald Williams. They published their work on backpropagation in 1986, in a paper titled "Learning Representations by Back-Propagating Errors." However, the concept of backpropagation was also developed independently by other researchers, including David Parker in 1985 and Yann LeCun in 1988.


Project Explanation

This project creates a LangGraph-based agent that can answer both general questions and mathematical queries.

The agent uses an LLM as the main reasoning component. Four custom tools were created: add, subtract, multiply, and divide. These tools are exposed to the LLM using LangChain's tool binding feature.

The LangGraph workflow contains two main nodes:

mathbot node: receives the user query and decides whether to answer directly or call a tool.
tools node: executes the selected mathematical function when the LLM requests a tool call.
The graph starts at the mathbot node. If the query requires a math operation, the conditional edge routes the flow to the tools node. After tool execution, the result is returned to the chatbot node so the LLM can generate a final natural-language answer. If no tool is needed, the graph ends directly after the mathbot response.

Division by zero is handled inside the divide function by returning an error message instead of crashing the program.